# 06. Robustness and the cost of look-ahead bias

Six alternative designs, each re-running the full 361-origin backtest:

| Variant | Change |
|---|---|
| `early_indicators_only` | no official output or trade data: retail, labour market, prices, money and markets only |
| `covid_in_training` | 2020Q1-2021Q2 kept in the training data |
| `rolling_window_40q` | train on the latest 40 quarters only |
| `one_se_rule` | penalties chosen by the one-standard-error rule instead of minimum CV error |
| `lookahead_latest_vintage_output` | today's revised monthly GDP and sector output instead of real-time output; still trained and judged on first estimates |
| `conventional_latest_vintage` | the usual published design: today's revised data as features, training target **and** yardstick |

In [1]:
import sys, warnings, pickle
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (9, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
T = ROOT / "outputs" / "tables"; FIG = ROOT / "outputs" / "figures"
from nowcast.config import HORIZON_LABELS

In [2]:
rob = pd.read_csv(T / "robustness.csv")
order = ["early_indicators_only", "covid_in_training", "rolling_window_40q", "one_se_rule"]
for v in order:
    d = rob[rob.variant == v]
    t = pd.concat({"variant": d.pivot(index="model", columns="h", values="rmse"),
                   "main": d.pivot(index="model", columns="h", values="main_rmse")}, axis=1)
    print(v); display(t.round(3))

early_indicators_only


variant                               main                            
h                -1      0      1      2      3     -1      0      1      2      3
model                                                                             
AR            0.629  0.495  0.485  0.504  0.468  0.629  0.495  0.485  0.504  0.468
Elastic net   0.565  0.515  0.538  0.510  0.408  0.549  0.495  0.497  0.485  0.414
Group lasso   0.584  0.493  0.514  0.493  0.455  0.568  0.475  0.493  0.474  0.354
Lasso         0.575  0.527  0.581  0.515  0.393  0.559  0.528  0.522  0.498  0.405
Ridge         0.546  0.484  0.506  0.474  0.420  0.559  0.477  0.484  0.453  0.383

covid_in_training


variant                               main                            
h                         -1      0      1      2      3     -1      0      1      2      3
model                                                                                      
AR                     0.577  0.493  0.605  0.618  0.760  0.629  0.495  0.485  0.504  0.468
Bridge                 0.672  0.510  0.696  0.330  0.273  0.590  0.476  0.467  0.364  0.320
Bridge + group lasso   0.995  1.396  0.652  0.379  0.255  0.617  0.548  0.530  0.409  0.343
Bridge + ridge         1.068  1.086  0.692  0.378  0.259  0.609  0.515  0.510  0.412  0.372
Elastic net            0.702  0.820  0.543  0.551  0.480  0.549  0.495  0.497  0.485  0.414
Group lasso            0.777  0.736  0.520  0.530  0.348  0.568  0.475  0.493  0.474  0.354
Lasso                  0.698  0.823  0.571  0.484  0.418  0.559  0.528  0.522  0.498  0.405
Ridge                  0.564  0.495  0.553  0.664  0.700  0.559  0.477  0.484  0.453  0.383

rolling_window_40q


variant                               main                            
h                         -1      0      1      2      3     -1      0      1      2      3
model                                                                                      
AR                     0.589  0.505  0.468  0.471  0.355  0.629  0.495  0.485  0.504  0.468
Bridge                 0.545  0.591  0.503  0.377  0.415  0.590  0.476  0.467  0.364  0.320
Bridge + group lasso   0.617  0.648  0.577  0.539  0.621  0.617  0.548  0.530  0.409  0.343
Bridge + ridge         0.610  0.594  0.524  0.458  0.628  0.609  0.515  0.510  0.412  0.372
Elastic net            0.570  0.478  0.495  0.465  0.408  0.549  0.495  0.497  0.485  0.414
Group lasso            0.556  0.478  0.471  0.444  0.384  0.568  0.475  0.493  0.474  0.354
Lasso                  0.576  0.520  0.517  0.465  0.419  0.559  0.528  0.522  0.498  0.405
Ridge                  0.541  0.478  0.449  0.425  0.348  0.559  0.477  0.484  0.453  0.383

one_se_rule


variant                               main                            
h                         -1      0      1      2      3     -1      0      1      2      3
model                                                                                      
AR                     0.629  0.495  0.485  0.504  0.468  0.629  0.495  0.485  0.504  0.468
Bridge                 0.590  0.476  0.467  0.364  0.320  0.590  0.476  0.467  0.364  0.320
Bridge + group lasso   0.587  0.492  0.446  0.353  0.320  0.617  0.548  0.530  0.409  0.343
Bridge + ridge         0.599  0.485  0.425  0.364  0.320  0.609  0.515  0.510  0.412  0.372
Elastic net            0.564  0.550  0.547  0.524  0.341  0.549  0.495  0.497  0.485  0.414
Group lasso            0.556  0.550  0.553  0.506  0.366  0.568  0.475  0.493  0.474  0.354
Lasso                  0.564  0.550  0.549  0.550  0.347  0.559  0.528  0.522  0.498  0.405
Ridge                  0.556  0.540  0.529  0.509  0.325  0.559  0.477  0.484  0.453  0.383

**Reading the variants** (RMSE against the first estimate):
* **Early indicators only.** Without output or trade data, ridge's error falls only from 0.55 to
  0.47 between the start and the end of the quarter, against 0.59 to 0.36 for the bridge. The
  late-quarter gains in the main design come from official output data.
* **COVID in training.** It cuts both ways. Models that lean on retail, labour-market, price
  and financial indicators get much worse early in the quarter: six extreme quarters dominate their
  fitted coefficients. The bridge-type models get *better* once output data arrive, because
  2020's huge output swings pin down the mapping from monthly output to GDP. The main design
  excludes COVID from training. That is the cautious choice for most models, and it slightly
  understates what the bridge could achieve after 2021.
* **Rolling 40-quarter window.** AR and the pure regularised models gain a little (up to 9%) from
  discarding old data. The bridge and the anchored models lose heavily, because their output
  coefficients are pinned down by the large swings of 2008-09, which a 10-year window eventually
  drops. The bridge is still the most accurate model at the end of the quarter.
* **One-standard-error rule.** Heavier penalties make the bridge-anchored models 6-17% more
  accurate, which brings them close to the plain bridge. The less weight the extra indicators get,
  the better. The pure regularised models mostly get worse. The conclusions do not depend on the
  tuning rule.

## Look-ahead bias in numbers

In [3]:
la = pd.read_csv(T / "lookahead.csv")
t = la.pivot_table(index=["model", "design"], columns="h", values="rmse").rename(columns=HORIZON_LABELS)
t.loc[["AR", "Bridge", "Ridge"]].round(3)

h                                                          Before quarter  End of month 1  End of month 2  End of month 3  Month after quarter
model  design                                                                                                                                 
AR     Conventional: revised data, trained and judged ...           0.714           0.507           0.629           0.629                0.749
       Real time, judged against first estimate (this ...           0.629           0.495           0.485           0.504                0.468
       Real time, judged against latest estimate                    0.733           0.613           0.572           0.580                0.366
       Revised output data, judged against first estimate           0.629           0.495           0.485           0.504                0.468
Bridge Conventional: revised data, trained and judged ...           0.656           0.496           0.592           0.400                0.169
       Real time, judged against first estimate (this ...           0.590           0.476           0.467           0.364                0.320
       Real time, judged against latest estimate                    0.683           0.570           0.519           0.427                0.255
       Revised output data, judged against first estimate           0.564           0.456           0.475           0.446                0.346
Ridge  Conventional: revised data, trained and judged ...           0.662           0.579           0.554           0.500                0.288
       Real time, judged against first estimate (this ...           0.559           0.477           0.484           0.453                0.383
       Real time, judged against latest estimate                    0.582           0.518           0.521           0.487                0.304
       Revised output data, judged against first estimate           0.556           0.501           0.486           0.473                0.432

In [4]:
def get(design, model, h):
    d = la[(la.design.str.startswith(design)) & (la.model == model) & (la.h == h)]
    return float(d.rmse.iloc[0])

rt, cv = "Real time, judged against first", "Conventional"
for h in [2, 3]:
    b_rt, a_rt = get(rt, "Bridge", h), get(rt, "AR", h)
    b_cv, a_cv = get(cv, "Bridge", h), get(cv, "AR", h)
    print(f"{HORIZON_LABELS[h]:<20} bridge RMSE: real time {b_rt:.3f} | conventional backtest {b_cv:.3f}"
          f"   ->  bridge vs AR: real time {1 - b_rt / a_rt:+.0%} | conventional {1 - b_cv / a_cv:+.0%}")

End of month 3       bridge RMSE: real time 0.364 | conventional backtest 0.400   ->  bridge vs AR: real time +28% | conventional +36%
Month after quarter  bridge RMSE: real time 0.320 | conventional backtest 0.169   ->  bridge vs AR: real time +32% | conventional +77%


Two findings:

1. **A conventional backtest flatters the late nowcast.** In the month after the quarter, today's
   revised monthly GDP for the quarter's first two months nearly adds up to today's quarterly GDP.
   A backtest built on revised data then reports an error about half the size that a forecaster could
   actually achieve, and a far larger advantage over the AR benchmark. Published nowcasting results
   that use latest-vintage data should be read with this in mind.
2. **Revised data are the wrong input for predicting the first estimate.** Swapping in revised
   output while still targeting the first estimate makes nowcasts *worse* at the end of the quarter.
   ONS builds its first estimate from output data as first published, not as later revised, so the
   real-time output series are the right inputs for the number markets react to.

Before any output data for the quarter exist (before the quarter, end of month 1), the choice of
vintage matters little. Look-ahead bias is concentrated where the hard data are.